# Lab 4 — The same agentic RAG in LangGraph (EP11 · framework lab)

In Lab 2 we wrote the plan → retrieve → evaluate → generate loop ourselves with `for` and `if` — the book writes the same loop with **LangGraph** (Listing 16.10).
This lab keeps every tool from Lab 2 and changes only **the loop**

| Lab 2 (by hand) | LangGraph |
|---|---|
| the variables `docs`, `tried`, `sub_queries` | `AgentState` (TypedDict) + the `operator.add` reducer |
| the plan / retrieve / evaluate / generate steps | four nodes — functions that take the state and return **only the keys they change** |
| `for iteration in range(max_iterations)` + `break` | the conditional edge `should_retrieve` |
| `print` inside the loop | `stream(stream_mode="updates")` |
| — | `recursion_limit`, the framework's own safety net against loops |

Every answer from the model comes from a real run.

> Source: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §16.7.4 Listing 16.10 — CC BY-SA 4.0

## 0. Setup

| You need | Details |
|---|---|
| Python 3.10 or later | or Google Colab, which already has it |
| An OpenRouter API key | the same key as Lab 1 (this lab costs under 1 US cent) — how to set up the key, and why we chose the model, is in Lab 1 |
| Packages | `langgraph==1.2.16` (the same as EP04, EP05, EP08), `openai`, `python-dotenv`, `pypdf`, `numpy`, `tokenizers` |

**Adding the API key**
- **Colab:** the 🔑 Secrets icon → name `OPENROUTER_API_KEY` → turn on *Notebook access*
- **Your own machine:** create a `.env` file next to the notebook with the line `OPENROUTER_API_KEY=sk-or-...`

⚠️ Never put the key in the code, and never share your `.env` file

In [ ]:
%pip install -q "langgraph==1.2.16" "openai>=1.50" python-dotenv "pypdf>=6" numpy "tokenizers==0.23.3"

## 1. Every tool from Lab 2

The next three cells are Lab 2's code, nothing new: the models + `ask_json` · the 141-chunk library + BM25 · three sources (keyword, vector, table of contents) + `grade`, `route`, `plan`, `evaluate`, `generate`

In [ ]:
import os, re, math, json, operator, urllib.request
from collections import Counter
from typing import TypedDict, Annotated
import numpy as np
from openai import OpenAI
from pypdf import PdfReader
from tokenizers import Tokenizer

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

MODEL = "google/gemma-4-26b-a4b-it"
EXTRA = {"reasoning": {"enabled": False},
         "provider": {"order": ["makora"], "allow_fallbacks": False}}
EMBED_MODEL = "qwen/qwen3-embedding-8b"
EMBED_EXTRA = {"provider": {"order": ["nebius"], "allow_fallbacks": False}}
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key(), max_retries=5)
COST = {"chat": 0.0, "embed": 0.0, "calls": 0}

def ask(prompt):
    r = client.chat.completions.create(model=MODEL, temperature=0, extra_body=EXTRA,
                                       messages=[{"role": "user", "content": prompt}])
    COST["chat"] += r.usage.cost
    COST["calls"] += 1
    return r.choices[0].message.content

def embed(texts):
    vectors = []
    for i in range(0, len(texts), 64):
        r = client.embeddings.create(model=EMBED_MODEL, input=texts[i:i + 64],
                                     extra_body=EMBED_EXTRA)
        COST["embed"] += r.usage.cost
        vectors += [d.embedding for d in r.data]
    V = np.array(vectors)
    return V / np.linalg.norm(V, axis=1, keepdims=True)

def ask_json(prompt, schema):
    for attempt in range(2):                            # one retry if the JSON comes back cut off (Lab 3)
        r = client.chat.completions.create(
            model=MODEL, temperature=0, extra_body=EXTRA, max_tokens=1500,
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_schema",
                             "json_schema": {"name": "reply", "strict": True, "schema": schema}})
        COST["chat"] += r.usage.cost
        COST["calls"] += 1
        try:
            return json.loads(r.choices[0].message.content)
        except json.JSONDecodeError:
            pass
    raise ValueError("the model did not return valid JSON")

def obj(**fields):
    return {"type": "object", "properties": fields,
            "required": list(fields), "additionalProperties": False}
TEXT = {"type": "string"}
def one_of(*values):
    return {"type": "string", "enum": list(values)}

In [ ]:
urllib.request.urlretrieve("https://arxiv.org/pdf/2606.24937v1", "hitchhikers-guide-v1.pdf")
reader = PdfReader("hitchhikers-guide-v1.pdf")
HEADER = "H. Roitman — The Hitchhiker’s Guide to Agentic AI: From Foundations to Systems\n"
pages = [{"page": n, "text": reader.pages[n - 1].extract_text().replace(HEADER, "")}
         for n in range(292, 375)]                      # chapters 15–19
tok = Tokenizer.from_pretrained(MODEL)

def chunk_page(page, W=512, O=64):
    ids = tok.encode(page["text"], add_special_tokens=False).ids
    return [{"page": page["page"], "text": tok.decode(ids[s:s + W])}
            for s in range(0, max(len(ids) - O, 1), W - O)]

chunks = [c for p in pages for c in chunk_page(p)]
E = embed([c["text"] for c in chunks])

def words(text):                                        # BM25 from EP10 Lab 2
    return re.findall(r"\w+", text.lower())
docs = [words(c["text"]) for c in chunks]
N, avgdl = len(docs), sum(len(d) for d in docs) / len(docs)
df = Counter(w for d in docs for w in set(d))
tfs = [Counter(d) for d in docs]
def idf(t):
    n = df.get(t, 0)
    return math.log((N - n + 0.5) / (n + 0.5) + 1)
def bm25_rank(query, k1=1.5, b=0.75):
    scores = [sum(idf(t) * tf[t] * (k1 + 1) / (tf[t] + k1 * (1 - b + b * len(d) / avgdl))
                  for t in words(query)) for d, tf in zip(docs, tfs)]
    return list(np.argsort(-np.array(scores)))
print(len(chunks), "chunks")

In [ ]:
def as_docs(order, source, k=3):
    return [{"source": source, "pages": [chunks[i]["page"]], "text": chunks[i]["text"]}
            for i in order[:k]]
def search_keyword(query):
    return as_docs(bm25_rank(query), "keyword_search")
def search_vector(query):
    return as_docs(list(np.argsort(-(E @ embed([query])[0]))), "vector_db")

def walk(outline, depth=0):                             # the table of contents as a tree
    for item in outline:
        if isinstance(item, list):
            yield from walk(item, depth + 1)
        else:
            yield depth, item.title, reader.get_destination_page_number(item) + 1
items = [x for x in walk(reader.outline) if x[0] >= 1 and 292 <= x[2] <= 374]
root = {"title": "The book, pages 292-374", "depth": 0, "start": 292, "end": 374, "children": []}
stack = [root]
for n, (depth, title, start) in enumerate(items):
    end = next((s for d, _, s in items[n + 1:] if d <= depth), 374)
    node = {"title": title, "depth": depth, "start": start, "end": max(start, end), "children": []}
    while stack[-1]["depth"] >= depth:
        stack.pop()
    stack[-1]["children"].append(node)
    stack.append(node)
PAGE = {p["page"]: p["text"] for p in pages}
MENU = obj(reasoning=TEXT, action=TEXT)
LEAF = obj(reasoning=TEXT, action=one_of("use", "back"))
def label(n):
    return f'{n["title"]} (pp. {n["start"]}-{n["end"]})'

def search_toc(query, max_steps=10):
    node, path, read = root, [root], []
    for step in range(1, max_steps + 1):
        if node["children"]:
            menu = "\n".join(f"[{i}] {label(c)}" for i, c in enumerate(node["children"]))
            out = ask_json(f"You are finding the part of a book that answers a question, using its table of contents.\n"
                           f"Question: {query}\nYou are at: {' > '.join(n['title'] for n in path)}\n"
                           f"Already read, did not help: {', '.join(read) or 'nothing'}\n\nSections:\n{menu}\n\n"
                           'Set "action" to the number of the section to open, or to "back" if none fits.', MENU)
        else:
            text = "\n".join(PAGE[p] for p in range(node["start"], node["end"] + 1))
            out = ask_json(f"Question: {query}\nYou opened: {label(node)}\n\n{text}\n\n"
                           'Set "action" to "use" if this text helps answer the question, or "back" if it does not.', LEAF)
        action = out["action"].strip("[] ")
        if action == "use":
            return [{"source": "table_of_contents", "text": text,
                     "pages": list(range(node["start"], node["end"] + 1))}]
        if action.isdigit() and int(action) < len(node["children"]):
            node = node["children"][int(action)]
            path.append(node)
        else:
            if not node["children"]:
                read.append(node["title"])
            if len(path) > 1:
                path.pop()
                node = path[-1]
    return []

SOURCES = {"keyword_search": search_keyword, "vector_db": search_vector, "table_of_contents": search_toc}

GRADE = obj(reasoning=TEXT, grade=one_of("correct", "ambiguous", "incorrect"))
def grade(query, doc):
    return ask_json(f"Question: {query}\n\nDocument (pages {doc['pages']}):\n{doc['text']}\n\n"
                    "Grade the document. correct: it contains the answer. "
                    "ambiguous: it is related but does not answer. "
                    "incorrect: it is not about the question.", GRADE)["grade"]

ROUTE = obj(reasoning=TEXT, source=one_of(*SOURCES), refined_query=TEXT)
def route(query, tried=()):
    if set(tried) >= set(SOURCES):                      # every source already tried: start over
        tried = ()
    note = (f"\nAlready tried, found nothing useful: {', '.join(tried)}. Pick another source."
            if tried else "")
    return ask_json(f'Given the query: "{query}"\nDecide which knowledge source to use:\n'
                    "- keyword_search: BM25 over the book's text; best for rare exact terms, acronyms and names\n"
                    "- vector_db: embedding search over the book's text; best for questions phrased in different words from the book\n"
                    "- table_of_contents: walk the book's table of contents; best when the question asks where in the book a topic is covered\n"
                    f"Return the source, a refined query for that source, and your reasoning.{note}", ROUTE)

BOOK = "Roitman, The Hitchhiker's Guide to Agentic AI"
LIBRARY = f"Our library is one book: {BOOK}, pages 292-374."
PLAN = obj(sub_queries={"type": "array", "items": TEXT})
def plan(query):
    return ask_json(f"{LIBRARY}\nSplit this question into the separate things we must look up in the book: "
                    "one sub-query per information need, and only one if it asks for one thing."
                    f"\n\nQuestion: {query}", PLAN)["sub_queries"]

EVAL = obj(reasoning=TEXT, sufficient={"type": "boolean"}, missing={"type": "array", "items": TEXT})
def evaluate(query, docs):
    context = "\n\n".join(f"[pages {d['pages']}]\n{d['text']}" for d in docs) or "(nothing yet)"
    return ask_json(f"Question: {query}\n\nRetrieved context:\n{context}\n\n"
                    "Does this context answer every part of the question? If a part is not answered yet, "
                    "list one sub-query per missing part; do not ask for extra detail.", EVAL)

SYSTEM_PROMPT = """You are a helpful assistant. Answer the question using ONLY
the provided context. If the context does not contain enough information,
say so explicitly. Cite your sources using [Doc N] notation."""
def generate(query, docs):
    context = "\n\n".join(f"[Doc {i+1}] (Source: {BOOK}, Pages: {d['pages']})\n{d['text']}"
                          for i, d in enumerate(docs))
    return ask(f"{SYSTEM_PROMPT}\n\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:")

## 2. State: everything the agent remembers on the way

In Lab 2 these were variables inside `agent()` — in LangGraph they become one `AgentState` that every node reads and writes.
A key with the **reducer** `operator.add` **appends** what a node returns instead of overwriting it

In [ ]:
class AgentState(TypedDict):
    query: str
    sub_queries: list[str]
    retrieved_docs: Annotated[list[dict], operator.add]
    tried: Annotated[list[str], operator.add]
    context_sufficient: bool
    answer: str
    iterations: int
    max_iterations: int

## 3. Four nodes

Each node takes the state and returns **a dict of the keys it changes** — LangGraph merges it into the state

In [ ]:
def plan_node(state):
    return {"sub_queries": plan(state["query"]), "iterations": 0}

def retrieve_node(state):
    new_docs, failed = [], []
    for sq in state["sub_queries"]:
        r = route(sq, state["tried"])
        found = SOURCES[r["source"]](r["refined_query"])
        good = [d for d in found if grade(sq, d) == "correct"]
        print(f"  retrieve: {r['source']:17} {len(good)}/{len(found)} correct  {sq[:45]!r}")
        if not good:
            failed.append(r["source"])
        new_docs += good
    return {"retrieved_docs": new_docs, "tried": failed,
            "iterations": state["iterations"] + 1}

def evaluate_node(state):
    check = evaluate(state["query"], state["retrieved_docs"])
    print(f"  evaluate: sufficient = {check['sufficient']}")
    return {"context_sufficient": check["sufficient"],
            "sub_queries": check["missing"] or state["sub_queries"]}

def generate_node(state):
    return {"answer": generate(state["query"], state["retrieved_docs"])}

In [ ]:
def should_retrieve(state):
    if state["context_sufficient"]:
        return "generate"
    if state["iterations"] >= state["max_iterations"]:
        return "generate"        # give up and generate with what we have
    return "retrieve"

## 4. Wire the nodes into a graph

`plan → retrieve → evaluate → (generate, or back to retrieve) → END` — as in Listing 16.10

In [ ]:
from langgraph.graph import StateGraph, END

workflow = StateGraph(AgentState)
for name, fn in [("plan", plan_node), ("retrieve", retrieve_node),
                 ("evaluate", evaluate_node), ("generate", generate_node)]:
    workflow.add_node(name, fn)
workflow.set_entry_point("plan")
workflow.add_edge("plan", "retrieve")
workflow.add_edge("retrieve", "evaluate")
workflow.add_conditional_edges("evaluate", should_retrieve,
                               {"retrieve": "retrieve", "generate": "generate"})
workflow.add_edge("generate", END)
agent = workflow.compile()
for e in agent.get_graph().edges:
    print(f"{e.source:9} -> {e.target:9} {'(conditional)' if e.conditional else ''}")

## 5. Run the agent

`stream(..., stream_mode="updates")` yields each node's update as it happens — so we see which nodes the graph walks through

In [ ]:
def start(query, max_iterations=3):
    return {"query": query, "max_iterations": max_iterations,
            "retrieved_docs": [], "tried": [], "iterations": 0}

for update in agent.stream(start("What is PLAID, and how does the textbook add randomness to retry waits?"),
                           stream_mode="updates"):
    print("node:", list(update))

In [ ]:
result = agent.invoke(start("What does the acronym PLAID stand for?"))
print(result["iterations"], "rounds ·", len(result["retrieved_docs"]), "documents")
print(result["answer"])

## 6. A pitfall in the book's code

The nodes in Listing 16.10 return `{**state, ...}` — **the whole state**, including the existing `retrieved_docs`.
But `retrieved_docs` has the `operator.add` reducer → LangGraph **appends the existing documents again**

In [ ]:
def book_evaluate_node(state):                         # written the way Listing 16.10 writes its nodes
    check = evaluate(state["query"], state["retrieved_docs"])
    return {**state, "context_sufficient": check["sufficient"]}

book = StateGraph(AgentState)
for name, fn in [("plan", plan_node), ("retrieve", retrieve_node),
                 ("evaluate", book_evaluate_node), ("generate", generate_node)]:
    book.add_node(name, fn)
book.set_entry_point("plan")
book.add_edge("plan", "retrieve")
book.add_edge("retrieve", "evaluate")
book.add_conditional_edges("evaluate", should_retrieve, {"retrieve": "retrieve", "generate": "generate"})
book.add_edge("generate", END)

for name, g in [("ours", agent), ("book style", book.compile())]:
    s = g.invoke(start("What is PLAID, and how does the textbook add randomness to retry waits?"))
    print(f"{name:10}", len(s["retrieved_docs"]), "documents, pages", [d["pages"][0] for d in s["retrieved_docs"]])

## 7. What the framework adds: `recursion_limit`

If you forget the ceiling, or set it too high, LangGraph has its own safety net: at most `recursion_limit` steps (25 by default), then it stops with `GraphRecursionError`

In [ ]:
from langgraph.errors import GraphRecursionError

try:
    agent.invoke(start("What does the acronym PLAID stand for?", max_iterations=100),
                 {"recursion_limit": 10})
except GraphRecursionError as e:
    print("stopped:", str(e)[:80])

In [ ]:
print(COST["calls"], "model calls ·", f"chat ${COST['chat']:.4f} · embed ${COST['embed']:.5f}")

## Summary

| Lab 2 (by hand) | LangGraph | What we saw in the lab |
|---|---|---|
| `docs += good` | the `operator.add` reducer | a node returns only the new documents; LangGraph appends them |
| the `for` loop + `break` | `should_retrieve` + a conditional edge | the two-part question finished in one round · the unanswerable one hit 3 rounds and said the context lacks it |
| `print` | `stream(stream_mode="updates")` | plan → retrieve → evaluate → generate |
| — | `recursion_limit` | stopped at 10 steps with `GraphRecursionError` — no answer for the user |

⚠️ **The book's code:** nodes returning `{**state, ...}` duplicate `retrieved_docs` (2 → 4 documents) — return only the keys you change

**Exercises**
1. Make `plan_node` and `retrieve_node` return `{**state, ...}` like the book too — how many times are the documents duplicated?
2. Add a checkpoint with `InMemorySaver` (EP05 Lab 5) and look at `get_state_history` for the question that loops three rounds
3. Split `retrieve_node` into two nodes, `route` and `search` — the graph gets longer, but what more does the stream tell us?

**Next — the EP11 recap**